# 🎙️ PodGen AI – Pipeline Test Notebook

Use this notebook to test each stage of the podcast generation pipeline independently.

## Setup
```bash
pip install -r backend/requirements.txt
```

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), 'backend'))

from dotenv import load_dotenv
load_dotenv('backend/.env')

GROQ_API_KEY = os.getenv('GROQ_API_KEY', '')
print('GROQ_API_KEY set:', bool(GROQ_API_KEY))

## 1. Content Extraction

In [ ]:
from services.content_extractor import ContentExtractor

extractor = ContentExtractor()

# Test URL extraction
url = 'https://en.wikipedia.org/wiki/Artificial_intelligence'
text = extractor.extract_from_url(url)
print(f'Extracted {len(text):,} chars')
print('---')
print(text[:500])

## 2. RAG – Chunking & Embedding

In [ ]:
from services.rag_service import RAGService

rag = RAGService()
n = rag.ingest(text)
print(f'Indexed {n} chunks')

results = rag.retrieve('what is machine learning?', top_k=3)
for i, r in enumerate(results):
    print(f'\n--- Chunk {i+1} ---')
    print(r[:300])

## 3. Groq LLM – Research & Summarise

In [ ]:
from services.groq_service import GroqService

groq = GroqService()

# Summarise extracted content
summary = groq.summarise(text, max_words=200)
print('=== SUMMARY ===')
print(summary)

In [ ]:
# Research a topic
research = groq.research_topic('The future of AI agents', audience='general')
print('=== RESEARCH ===')
print(research[:1000])

## 4. Script Generation

In [ ]:
context = rag.get_summary_context(max_chars=3000)

script = groq.generate_script(
    context=context,
    topic='Artificial Intelligence',
    style='educational',
    audience='general',
    tone='conversational',
    duration_minutes=5,
    host_name='Alex',
    guest_name='Jordan',
)

print('=== SCRIPT EXCERPT ===')
print(script[:1500])

## 5. Metadata Generation

In [ ]:
import json
metadata = groq.generate_metadata(script, 'Artificial Intelligence')
print(json.dumps(metadata, indent=2))

## 6. Quality Scoring

In [ ]:
score = groq.score_script(script)
print(json.dumps(score, indent=2))

## 7. Script Parsing

In [ ]:
from services.tts_service import parse_script

segments = parse_script(script)
print(f'Parsed {len(segments)} speech segments')
for speaker, text in segments[:5]:
    print(f'\n[{speaker}]: {text[:120]}…')

## 8. TTS Audio Generation

In [ ]:
# Note: set TTS_ENGINE=gtts (free) or TTS_ENGINE=elevenlabs
import os
os.makedirs('audio_output', exist_ok=True)

from services.tts_service import TTSService

tts = TTSService()

def progress(pct):
    print(f'Audio: {pct}%', end='\r')

audio_file = tts.generate_audio(
    script=script,
    job_id='test_001',
    host_name='HOST',
    guest_name='GUEST',
    progress_callback=progress,
)

print(f'\nAudio saved: audio_output/{audio_file}')

In [ ]:
# Play audio inline in Jupyter
from IPython.display import Audio
Audio(f'audio_output/{audio_file}')

## 9. Full Pipeline End-to-End
Run the complete pipeline on a topic.

In [ ]:
import asyncio
from services.job_manager import JobManager
from services.pipeline import PodcastPipeline

jm = JobManager()
pipeline = PodcastPipeline(jm)

job_id = 'notebook_test_001'
jm.create_job(job_id, 'topic', {'topic': 'Climate change solutions'})

config = {
    'style': 'educational',
    'audience': 'general',
    'tone': 'conversational',
    'duration_minutes': 5,
    'host_name': 'Alex',
    'guest_name': 'Jordan',
    'host_personality': 'curious and engaging',
    'guest_personality': 'knowledgeable and enthusiastic',
}

await pipeline.run(
    job_id=job_id,
    input_type='topic',
    content='Climate change solutions',
    config=config,
)

job = jm.get_job(job_id)
print('Status:', job['status'])
print('Title:', job['title'])
print('Audio URL:', job['audio_url'])
print('Quality Score:', job.get('quality_score', {}).get('overall'))